# BLIP-3 Baseline Visualization Demo

Notebook ini memvisualisasikan hasil baseline pilot VQAv2 sebanyak 100 pertanyaan.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

BASE = Path('/workspace') if Path('/workspace/outputs/metrics/baseline_pilot.json').exists() else Path('.')
metrics = json.loads((BASE / 'outputs/metrics/baseline_pilot.json').read_text())
df = pd.read_json(BASE / 'outputs/predictions/baseline_pilot.json')
df.head()

## 1. Ringkasan baseline

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Questions', 'Accuracy', 'Mean latency (s)', 'Latency SD (s)', 'Peak memory (GB)'],
    'Value': [metrics['n'], metrics['accuracy'], metrics['mean_latency_s'], metrics['latency_sd_s'], metrics['peak_memory_gb']]
})
display(summary.style.format({'Value': '{:.4f}'}))

## 2. Accuracy berdasarkan answer type

In [ ]:
accuracy_by_type = pd.Series(metrics['accuracy_by_answer_type']).sort_values(ascending=False)
ax = accuracy_by_type.plot(kind='bar', figsize=(7, 4), color='#4472C4', ylim=(0, 1), title='Baseline accuracy by answer type')
ax.set_ylabel('Accuracy')
ax.set_xlabel('Answer type')
ax.bar_label(ax.containers[0], fmt='%.3f')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Distribusi latency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['latency_s'].plot(kind='hist', bins=15, ax=axes[0], color='#70AD47', edgecolor='white')
axes[0].set_title('Latency distribution')
axes[0].set_xlabel('Latency (seconds)')
df['latency_s'].plot(ax=axes[1], color='#ED7D31')
axes[1].axhline(2.0, color='red', linestyle='--', label='2-second criterion')
axes[1].set_title('Latency by question order')
axes[1].set_xlabel('Question index')
axes[1].set_ylabel('Latency (seconds)')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Contoh gambar, pertanyaan, dan prediksi

In [ ]:
image_dir = BASE / 'data/vqav2/images/val2014'
for _, row in df.head(6).iterrows():
    image_path = image_dir / f"COCO_val2014_{int(row['image_id']):012d}.jpg"
    print(f"QID {int(row['question_id'])} | {row['answer_type']} | latency={row['latency_s']:.3f}s")
    print('Question:', row['question'])
    print('Prediction:', row['prediction'])
    if image_path.exists():
        display(Image.open(image_path).resize((320, 240)))
    print('-' * 80)

## 5. Catatan interpretasi

Hasil ini adalah pilot 100 pertanyaan, bukan hasil eksperimen tesis final. Latency diukur pada RTX 5090 32 GB dan harus diperlakukan sebagai perbandingan hardware-specific.